# 05. AI 3줄 요약 (GPT-4o-mini)

이 Notebook에서 하는 일:
1. 포트폴리오 현황 데이터 수집 (자산, 위험점수, 인출 정보)
2. GPT-4o-mini로 한국어 3줄 요약 생성
3. API 실패 시 룰 기반 폴백(fallback) 자동 적용
4. 요약 결과 DB 저장
5. 과거 요약 이력 조회

In [1]:
# 공통 설정
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date
from openai import OpenAI

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
WITHDRAWAL_CFG  = CONFIG['withdrawal']
USER_NAME       = CONFIG['user']['name']

api_key = os.getenv('OPENAI_API_KEY', '')
client  = OpenAI(api_key=api_key) if api_key else None

print(f"사용자: {USER_NAME}")
print(f"OpenAI API: {'연결됨' if client else '키 없음 — 룰 기반 폴백 사용'}")

사용자: 종헌
OpenAI API: 연결됨


In [2]:
# 자산 데이터 로드
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print('※ 샘플 데이터 사용 중')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
type_sum   = df.groupby('asset_type')['current_value'].sum()
total      = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)

print(f'총 자산: {total:,.0f}원  |  버킷1: {b1:,.0f}  버킷2: {b2:,.0f}  버킷3: {b3:,.0f}')

※ 샘플 데이터 사용 중
총 자산: 67,550,000원  |  버킷1: 35,000,000  버킷2: 31,130,000  버킷3: 1,420,000


In [3]:
# DB에서 최신 위험 점수 로드
db_path = PROJECT_ROOT / 'portfolio.db'
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()

cur.execute('SELECT total_score, cash_score, seq_score, conc_score, level FROM risk_scores ORDER BY date DESC LIMIT 1')
risk_row = cur.fetchone()

cur.execute('SELECT amount, guardrail_applied FROM withdrawal_log ORDER BY date DESC LIMIT 1')
wd_row = cur.fetchone()

conn.close()

if risk_row:
    total_score, cash_score, seq_score, conc_score, risk_level = risk_row
else:
    total_score, cash_score, seq_score, conc_score, risk_level = 0, 0, 0, 0, 'green'
    print('※ 위험 점수 없음 — 03_risk_score.ipynb를 먼저 실행하세요')

if wd_row:
    last_withdrawal, guardrail_applied = wd_row
else:
    last_withdrawal, guardrail_applied = MONTHLY_EXPENSE, 0

months_covered  = b1 / MONTHLY_EXPENSE if MONTHLY_EXPENSE > 0 else 0
equity_ratio    = (type_sum.get('equity', 0) + type_sum.get('income', 0)) / total if total > 0 else 0
cash_ratio      = type_sum.get('cash', 0) / total if total > 0 else 0
bond_ratio      = (type_sum.get('bond', 0) + type_sum.get('tdf', 0)) / total if total > 0 else 0

level_map = {'green': '녹색(안전)', 'yellow': '황색(주의)', 'red': '적색(위험)'}
print(f'위험 등급: {level_map.get(risk_level, risk_level)}  |  종합 점수: {total_score:.1f}점')
print(f'현금 {months_covered:.1f}개월치  |  주식 비중 {equity_ratio*100:.1f}%')

위험 등급: 황색(주의)  |  종합 점수: 28.0점
현금 14.0개월치  |  주식 비중 2.1%


## 1. 룰 기반 폴백(Fallback) 요약 생성

OpenAI API 호출 전, 규칙 기반으로 요약을 먼저 만들어 놓습니다.  
API 실패 시 이 요약이 자동으로 사용됩니다.

In [4]:
def rule_based_summary(total, b1, months_covered, equity_ratio, total_score,
                       risk_level, last_withdrawal, monthly_expense, guardrail_applied):
    """위험 점수와 포트폴리오 현황을 바탕으로 룰 기반 3줄 요약 생성"""

    # 줄 1: 전체 현황
    level_txt = {'green': '안전', 'yellow': '주의', 'red': '위험'}.get(risk_level, '확인 필요')
    line1 = (f"총 자산 {total/1e8:.2f}억원, 위험 등급 {level_txt}({total_score:.0f}점)으로 "
             f"{'현재 포트폴리오는 안정적입니다.' if risk_level == 'green' else '점검이 필요합니다.'}")

    # 줄 2: 현금성 자산
    if months_covered >= 12:
        line2 = f"현금성 자산은 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 충분히 확보되어 있습니다."
    elif months_covered >= 6:
        line2 = f"현금성 자산이 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 12개월치 확보를 권장합니다."
    else:
        line2 = f"현금성 자산이 {months_covered:.0f}개월치({b1/1e8:.2f}억원)로 부족합니다. 즉시 보충을 권장합니다."

    # 줄 3: 이번 달 인출
    guard_txt = ' (가드레일 하향 적용)' if guardrail_applied else ''
    if last_withdrawal <= monthly_expense:
        line3 = (f"이번 달 인출 권장액은 {last_withdrawal:,.0f}원{guard_txt}으로, "
                 f"{'생활비 범위 내 유지 중입니다.' if last_withdrawal >= monthly_expense * 0.9 else '생활비 조정을 권장합니다.'}")
    else:
        line3 = (f"이번 달 인출 권장액은 {last_withdrawal:,.0f}원{guard_txt}으로, "
                 f"현재 생활비({monthly_expense:,.0f}원) 범위 내입니다.")

    return f"{line1}\n{line2}\n{line3}"


fallback_summary = rule_based_summary(
    total, b1, months_covered, equity_ratio, total_score,
    risk_level, last_withdrawal, MONTHLY_EXPENSE, guardrail_applied
)

print('=== 룰 기반 폴백 요약 (미리보기) ===')
print(fallback_summary)

=== 룰 기반 폴백 요약 (미리보기) ===
총 자산 0.68억원, 위험 등급 주의(28점)으로 점검이 필요합니다.
현금성 자산은 14개월치(0.35억원)로 충분히 확보되어 있습니다.
이번 달 인출 권장액은 225,167원으로, 생활비 조정을 권장합니다.


## 2. GPT-4o-mini AI 요약 생성

In [5]:
def build_prompt(total, b1, b2, b3, months_covered, equity_ratio, cash_ratio, bond_ratio,
                 total_score, cash_score, seq_score, conc_score, risk_level,
                 last_withdrawal, monthly_expense, guardrail_applied, user_name):
    """GPT에 전달할 프롬프트 생성"""
    level_txt = {'green': '녹색(안전)', 'yellow': '황색(주의)', 'red': '적색(위험)'}.get(risk_level, risk_level)

    prompt = f"""당신은 은퇴자 {user_name}님의 포트폴리오를 관리하는 AI 자문가입니다.
아래 데이터를 바탕으로 이번 달 포트폴리오 현황을 친근하고 명확한 한국어로 정확히 3줄로 요약해 주세요.

**포트폴리오 현황 ({date.today().strftime('%Y년 %m월')})**
- 총 자산: {total/1e8:.2f}억원
- 버킷 1 (현금성): {b1/1e8:.2f}억원 ({months_covered:.1f}개월치 생활비)
- 버킷 2 (채권/TDF): {b2/1e8:.2f}억원
- 버킷 3 (성장/인컴): {b3/1e8:.2f}억원
- 자산 비중: 현금 {cash_ratio*100:.1f}% / 채권 {bond_ratio*100:.1f}% / 주식형 {equity_ratio*100:.1f}%

**위험 점수**
- 종합 위험 점수: {total_score:.1f}점 / 100점  등급: {level_txt}
- 현금 위험: {cash_score:.0f}점  순서 위험: {seq_score:.0f}점  집중 위험: {conc_score:.0f}점

**인출 정보**
- 이번 달 권장 인출액: {last_withdrawal:,.0f}원
- 월 생활비: {monthly_expense:,.0f}원
- 가드레일 적용: {'예 (인출 하향 조정)' if guardrail_applied else '아니오 (정상)'}

**요약 규칙**
1. 정확히 3줄, 각 줄은 완결된 문장
2. 첫 줄: 전체 현황 및 위험 등급
3. 둘째 줄: 현금성 자산 충분성 및 필요 조치
4. 셋째 줄: 이번 달 인출 권장 및 주의사항
5. 숫자는 억원 단위로 표기, 전문 용어는 쉽게 설명
6. 위험 등급이 황색/적색이면 구체적 행동 권고 포함"""

    return prompt


def generate_ai_summary():
    """GPT-4o-mini로 요약 생성, 실패 시 룰 기반 폴백 반환"""
    if not client:
        print('API 키 없음 → 룰 기반 폴백 사용')
        return fallback_summary, 'fallback'

    prompt = build_prompt(
        total, b1, b2, b3, months_covered, equity_ratio, cash_ratio, bond_ratio,
        total_score, cash_score, seq_score, conc_score, risk_level,
        last_withdrawal, MONTHLY_EXPENSE, guardrail_applied, USER_NAME
    )

    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': '당신은 은퇴 자산관리 전문 AI입니다. 항상 한국어로 답변합니다.'},
                {'role': 'user', 'content': prompt}
            ],
            max_tokens=300,
            temperature=0.4
        )
        summary = response.choices[0].message.content.strip()
        return summary, 'gpt-4o-mini'
    except Exception as e:
        print(f'API 오류: {e}')
        print('→ 룰 기반 폴백 자동 적용')
        return fallback_summary, 'fallback'


print('GPT-4o-mini 요약 생성 중...')
summary, source = generate_ai_summary()

print()
print(f'=== AI 포트폴리오 요약 ({date.today().strftime("%Y년 %m월 %d일")}) ===')
print(f'생성 방식: {source}')
print()
print(summary)

GPT-4o-mini 요약 생성 중...

=== AI 포트폴리오 요약 (2026년 05월 20일) ===
생성 방식: gpt-4o-mini

현재 포트폴리오의 총 자산은 0.68억원이며, 위험 점수는 28.0점으로 황색(주의) 등급입니다. 현금성 자산이 0.35억원으로 14개월치 생활비를 충분히 커버하고 있으나, 채권과 주식 비중이 낮아 분산 투자 필요성이 있습니다. 이번 달 권장 인출액은 0.0225억원이며, 인출 시 자산 분산을 고려해 주의가 필요합니다.


## 3. 요약 DB 저장

In [6]:
today   = str(date.today())
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()

# 오늘 이미 저장된 AI 요약이 있으면 업데이트
cur.execute("SELECT id FROM recommendations WHERE date=? AND rule_id='ai_summary'", (today,))
existing = cur.fetchone()

if existing:
    cur.execute("""
        UPDATE recommendations SET message=?, status='completed'
        WHERE date=? AND rule_id='ai_summary'
    """, (summary, today))
    print(f'오늘({today}) AI 요약 업데이트 완료')
else:
    cur.execute("""
        INSERT INTO recommendations (date, rule_id, message, status)
        VALUES (?, 'ai_summary', ?, 'completed')
    """, (today, summary))
    print(f'오늘({today}) AI 요약 저장 완료')

conn.commit()
conn.close()
print(f'생성 방식: {source}')

오늘(2026-05-20) AI 요약 저장 완료
생성 방식: gpt-4o-mini


## 4. 과거 요약 이력 조회

In [7]:
conn = sqlite3.connect(db_path)
history = pd.read_sql_query("""
    SELECT date, message FROM recommendations
    WHERE rule_id = 'ai_summary'
    ORDER BY date DESC LIMIT 6
""", conn)
conn.close()

if history.empty:
    print('저장된 AI 요약이 없습니다.')
else:
    print('=== 최근 AI 요약 이력 ===')
    print()
    for _, row in history.iterrows():
        print(f'[{row["date"]}]')
        print(row['message'])
        print()

=== 최근 AI 요약 이력 ===

[2026-05-20]
현재 포트폴리오의 총 자산은 0.68억원이며, 위험 점수는 28.0점으로 황색(주의) 등급입니다. 현금성 자산이 0.35억원으로 14개월치 생활비를 충분히 커버하고 있으나, 채권과 주식 비중이 낮아 분산 투자 필요성이 있습니다. 이번 달 권장 인출액은 0.0225억원이며, 인출 시 자산 분산을 고려해 주의가 필요합니다.



---

## 완료!

- GPT-4o-mini가 포트폴리오를 3줄로 요약했습니다.
- API 오류 시 룰 기반 요약이 자동으로 대체됩니다.

**다음 단계**: `06_dashboard.ipynb` — 통합 대시보드 (Voilà로 실행)